# Appendix B — Delivery hours, DST and strip arithmetic

Referenced from Chapters 1.1, 1.2 and 2.6.

## B.1 The gas day

In continental Europe the **gas day** runs from 06:00 to 06:00 local time (CET/CEST). A delivery month
therefore begins at 06:00 on the 1st and ends at 06:00 on the 1st of the following month. Because both
endpoints are in *local* time, the month's length in hours is affected by the two clock changes:

* **Last Sunday of March**: clocks go forward 02:00 → 03:00; the day has 23 hours. **March = 743 h**
  (31 × 24 − 1).
* **Last Sunday of October**: clocks go back 03:00 → 02:00; the day has 25 hours. **October = 745 h**
  (31 × 24 + 1).

Every other month is $24 \times \text{days}$: 744 (31 days), 720 (30 days), 672 (February, 696 in leap
years).

## B.2 Why it matters for a hedging engine

A lot is **1 MW in every hour**. Energy $=$ power × hours, so lot MWh $= H_{y,m}$. Consequences:

1. Lot sizing (eq. 1.2) must use the *contract's* hours, not a constant 720 or 744.
2. Strip decomposition (eq. 1.6) must weight by hours or the monthly MWh will not sum to the strip MWh.
3. Tick value $= 0.005 \times H$ differs by month (eq. 1.3).
4. P&L attribution across a roll is only exact if both legs' MWh are hours-consistent.

The implementation in `gashedge.contracts.delivery_hours` builds the two timezone-aware endpoints with
`zoneinfo` and takes the difference in UTC — robust to any future DST rule change as long as the OS
tz database is current.

In [1]:
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / "gashedge").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from datetime import date
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from gashedge import get_logger
from gashedge.plotting import setup_style
from gashedge.contracts import *
setup_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
log = get_logger("appendixB")
log.info("Environment ready")

2026-09-13 16:47:22.171 | INFO    | appendixB | Environment ready


In [2]:
rows = []
for y in (2026, 2027, 2028):
    for m in range(1, 13):
        start, end = delivery_period(y, m)
        rows.append({"year": y, "month": m, "start_local": start.isoformat(), "end_local": end.isoformat(),
                     "hours": delivery_hours(y, m), "naive_24xdays": 24 * pd.Period(f"{y}-{m:02d}").days_in_month})
tbl = pd.DataFrame(rows)
tbl["dst_adjustment"] = tbl.hours - tbl.naive_24xdays
log.info("Months with DST adjustment: %s", tbl[tbl.dst_adjustment != 0][["year", "month", "hours"]].to_dict("records"))
tbl[tbl.dst_adjustment != 0]

2026-09-13 16:47:22.176 | INFO    | appendixB | Months with DST adjustment: [{'year': 2026, 'month': 3, 'hours': 743}, {'year': 2026, 'month': 10, 'hours': 745}, {'year': 2027, 'month': 3, 'hours': 743}, {'year': 2027, 'month': 10, 'hours': 745}, {'year': 2028, 'month': 3, 'hours': 743}, {'year': 2028, 'month': 10, 'hours': 745}]


,year,month,start_local,end_local,hours,naive_24xdays,dst_adjustment
2,2026,3,2026-03-01T06:00:00+01:00,2026-04-01T06:00:00+02:00,743,744,-1
9,2026,10,2026-10-01T06:00:00+02:00,2026-11-01T06:00:00+01:00,745,744,1
14,2027,3,2027-03-01T06:00:00+01:00,2027-04-01T06:00:00+02:00,743,744,-1
21,2027,10,2027-10-01T06:00:00+02:00,2027-11-01T06:00:00+01:00,745,744,1
26,2028,3,2028-03-01T06:00:00+01:00,2028-04-01T06:00:00+02:00,743,744,-1
33,2028,10,2028-10-01T06:00:00+02:00,2028-11-01T06:00:00+01:00,745,744,1


## B.3 Strip arithmetic — proof of the hours-weighted average

Let a strip cover months $i=1..n$ with hours $H_i$ and prices $P_i$. One lot of the strip delivers 1 MW in
every hour of every month, i.e. $H_i$ MWh in month $i$. Absence of arbitrage between the strip and its
monthly legs requires equal value:

$$
P_{\text{strip}}\sum_i H_i = \sum_i P_i H_i
\quad\Longrightarrow\quad
P_{\text{strip}} = \frac{\sum_i H_i P_i}{\sum_i H_i} .
\tag{B.1}
$$

If instead the strip were priced as the *equal-weighted* mean $\bar P = \frac1n\sum P_i$, a trader could
buy the cheaper of {strip, legs} and sell the other; the profit per MW is

$$
\Big|\bar P - P_{\text{strip}}\Big| \sum_i H_i
= \Big|\sum_i P_i\Big(\tfrac{1}{n} - \tfrac{H_i}{\sum_j H_j}\Big)\Big|\sum_i H_i ,
\tag{B.2}
$$

which is non-zero whenever prices are not constant across months and hours differ. Equation (B.2) is
the "bias" column computed in Chapter 1.2.

## B.4 Decomposing a strip position into months

A strip position of $Q$ MWh over months with total hours $\sum_i H_i$ is $q = Q/\sum_i H_i$ MW, and hence
$Q_i = q H_i$ MWh in month $i$ (eq. 1.6). Check: $\sum_i Q_i = q\sum_i H_i = Q$. The equal split
$Q/n$ is wrong by $Q\,(1/n - H_i/\sum_j H_j)$ in month $i$.

## B.5 Season and quarter definitions

| strip | months | typical hours (non-leap) |
|---|---|---|
| Q1 | Jan, Feb, Mar | 744 + 672 + 743 = 2159 |
| Q2 | Apr, May, Jun | 720 + 744 + 720 = 2184 |
| Q3 | Jul, Aug, Sep | 744 + 744 + 720 = 2208 |
| Q4 | Oct, Nov, Dec | 745 + 720 + 744 = 2209 |
| Summer | Apr – Sep | 4392 |
| Winter | Oct – Mar (next year) | 4368 (4392 in a leap year) |
| Calendar | Jan – Dec | 8760 (8784 in a leap year) |

Note Q4 has 50 more hours than Q1 — one more day in October and December than in February, plus the DST
hour. A "Q4/Q1 spread" of 1 lot each is therefore *not* MWh-neutral (Chapter 1.2, Q2).

In [3]:
checks = {
    "Q1-27": sum(delivery_hours(y, m) for y, m in quarter_months(2027, 1)),
    "Q4-26": sum(delivery_hours(y, m) for y, m in quarter_months(2026, 4)),
    "Sum-27": sum(delivery_hours(y, m) for y, m in season_months(2027, "SUM")),
    "Win-26": sum(delivery_hours(y, m) for y, m in season_months(2026, "WIN")),
    "Win-27 (contains Feb-28, leap)": sum(delivery_hours(y, m) for y, m in season_months(2027, "WIN")),
    "Cal-27": sum(delivery_hours(y, m) for y, m in calendar_months(2027)),
    "Cal-28 (leap)": sum(delivery_hours(y, m) for y, m in calendar_months(2028)),
}
pd.Series(checks, name="hours")

Q1-27                             2159
Q4-26                             2209
Sum-27                            4392
Win-26                            4368
Win-27 (contains Feb-28, leap)    4392
Cal-27                            8760
Cal-28 (leap)                     8784
Name: hours, dtype: int64

---
◀ [Previous](A_least_squares.ipynb) · [Contents](../00_introduction.ipynb) · [Next ▶](C_notation_pca_references.ipynb)